## Advanced Machine Learning
### Practical 3 - Self-Supervised Learning

In this exercise we'll use self-supervised pretext learning to learn high-quality feature representations which can be adapted to a downstream task.

### Objective

Our objective:
- Maximise validation accuracy on a small labelled dataset (the downstream task)
- with the help of a large dataset which we use for pre-training

This setting will be familiar. It is essentially the transfer learning setting from Week 1, where we had:
- A large labelled "source" dataset, which we pre-train on to obtain useful features.
- A small unlabelled "target" dataset, which we fine-tune on as our actual target objective.

In fact, we will use the same datasets as our basis. However, there is one difference:
- This time, the source data is **completely unlabelled**
- So we must teach the model useful features without explicit annotations in the data itself
- This is the *self-supervised* paradigm, in its role as **unsupervised pre-training** for transfer learning

This is also similar to the *semi-supervised* exercise (self-training with pseudo-labels), where:
- We had used a large unlabelled dataset to augment learning on a small, labelled dataset
  - But in that one, the two datasets were from the same domain and optimised for the same task
- This time, in the self-supervised setting, our two datasets come from different domains, and represent different tasks!

Though here we have one advantage: The two datasets come from related distributions. Even though they don't share any classes, they share a lot of overlapping visual features that can still be re-used from one to the other. This is what we'll exploit when we pre-train our self-supervised model.

### Plan

Step-by-step:

1. We choose a small labelled dataset to define our **target task**. This is the CIFAR-100 subset scheme we've used before. It's what we want to maximise our validation accuracy on.

2. As a **baseline**, we will train a supervised model only on this data, to establish our lower-bound performance without self-supervised learning.

3. Then, we will choose a large unlabelled **source dataset** to learn from. In this case, it is CIFAR-10, with all of the labels destroyed.

4. You will define your own **pretext task** that can be derived from this unlabelled data. This is the key idea of self-supervised learning: creating useful labels from unsupervised data.

5. We will **pre-train** a model on this pretext task with the unlabelled data, hoping for stable convergence and good feature learning.

6. Then, we will fine-tune this model on the **target task**. Remember that this means replacing the task-specific head of our pre-trained model with a new one for the new task. What we're keeping is just the feature encoder **backbone** that was pre-trained.

If the pretext task is good, and the pre-training went well, we should see a large improvement over the baseline performance. Just like in transfer learning, but unsupervised.


#### How to run this practical

If you don't have a GPU on your machine, you can download this notebook and upload it to [Kaggle](https://www.kaggle.com/code) or [Colab](https://colab.research.google.com/) to use their GPU resources. 
- You'll need to toggle on GPU acceleration in whichever one you use. On Kaggle you have to toggle on internet access as well.

In [ ]:
# set up the environment and install any missing packages
!pip install torchinfo

## basic imports
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import models, datasets, transforms
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import PIL
from PIL.Image import Image

from tqdm import tqdm
from IPython.display import display, clear_output
from torchinfo import summary

import os, gdown

In [ ]:
### usual helpers for logging and visualisation:

class TrainingMetrics:
    def __init__(self):
        # initialise with empty lists for metric tracking:
        self.train_loss = []
        self.val_loss = []
        self.train_acc = []
        self.val_acc = []

        # for tracking unsupervised loss later:
        self.unsup_loss = []

        self.step = 0
        self.epoch = 0
        self.epoch_steps = [0] # assume validation happens after every epoch

        # for plotting:
        self.fig = None
        self.plot_axes = None

        self.iter_steps = [] # steps at which training iteration changes

    def log_train(self, loss:float, acc:float, unsup_loss:float=None):
        """record batch-wise training metrics at each step (and increment step)"""
        assert isinstance(loss, float) and isinstance(acc, float), f"log_train expects float inputs but got: {type(loss)} and {type(acc)}"
        self.train_loss.append(loss)
        self.train_acc.append(acc)

        if unsup_loss is not None:
            self.unsup_loss.append(unsup_loss)

        self.step += 1

    def log_val(self, loss:float, acc:float):
        """record total val metrics at end of epoch (and increment epoch)"""
        assert isinstance(loss, float) and isinstance(acc, float), f"log_val expects float inputs but got: {type(loss)} and {type(acc)}"
        assert self.step not in self.epoch_steps, f"Validation metrics already logged for epoch {self.epoch} - you should submit only the aggregate metrics at end of epoch"""
        self.val_loss.append(loss)
        self.val_acc.append(acc)

        self.epoch_steps.append(self.step)
        self.epoch += 1

    @property
    def epoch_loss(self):
        """report the training and validation loss over the previous epoch"""
        cur_epoch_step = self.epoch_steps[-1]
        last_epoch_step = self.epoch_steps[-2]
        epoch_train_loss = self.train_loss[last_epoch_step:cur_epoch_step]
        epoch_val_loss = self.val_loss[-1]
        return np.mean(epoch_train_loss), np.mean(epoch_val_loss)

    @property        
    def epoch_acc(self):
        """report the training and validation accuracy over the previous epoch"""
        cur_epoch_step = self.epoch_steps[-1]
        last_epoch_step = self.epoch_steps[-2]
        epoch_train_acc = self.train_acc[last_epoch_step:cur_epoch_step]
        epoch_val_acc = self.val_acc[-1]
        return np.mean(epoch_train_acc), np.mean(epoch_val_acc)

    @property
    def best_val_acc(self):
        return np.max(self.val_acc)

    @property
    def best_val_loss(self):
        return np.min(self.val_loss)        
    
    def plot(self,
             title:str = None, # optional figure title
             baseline_accs:dict[str:float] = None, # optional named val accuracy baselines to compare to
             baseline_metrics   = None, # optional TrainingMetrics object for comparison curves
             smooth_alpha:float = 0.1,  # smoothing parameter for train loss
             live:bool    = True, # for animated plotting during training (slightly experimental)
             max_step:int = 0,    # expected number of total training steps (for animated plotting)
             ):
        
        if self.fig is None:
            # make two subplots for loss and accuracy:
            self.fig, self.plot_axes = plt.subplots(1,2)
        loss_ax, acc_ax = self.plot_axes
        
        x_step = np.arange(0, self.step) # x axis step values
        ### plot loss on the left
        # train loss is plotted at every step (smoothed with ewm)
        smooth_train_loss = pd.Series(self.train_loss).ewm(alpha=smooth_alpha).mean()
        smooth_train_acc = pd.Series(self.train_acc).ewm(alpha=smooth_alpha).mean()
        loss_ax.clear()
        loss_ax.plot(x_step, smooth_train_loss, c='tab:orange', linestyle=':', label='train loss' if len(self.unsup_loss) == 0 else 'supervised loss')
        # if we have been tracking consistency loss, plot that too:
        if len(self.unsup_loss) > 0 and len(self.unsup_loss) == len(self.train_loss):
            smooth_unsup_loss = pd.Series(self.unsup_loss).ewm(alpha=smooth_alpha).mean()
            loss_ax.plot(x_step, smooth_unsup_loss, c='tab:green', linestyle=':', label='unsupervised loss')
        
        # val loss is plotted at every epoch:
        loss_ax.plot(self.epoch_steps[1:], self.val_loss, c='tab:orange', linestyle='-', label='val loss')
        loss_ax.set_xlabel('Training step'); loss_ax.set_ylabel('Loss')
        loss_ax.legend()
        
        ### plot accuracy on the right
        acc_ax.clear()
        acc_ax.plot(x_step, smooth_train_acc, c='tab:blue', linestyle=':', label='train acc')
        acc_ax.plot(self.epoch_steps[1:], self.val_acc, c='tab:blue', linestyle='-', label='val acc')
        acc_ax.legend(); acc_ax.set_xlabel('Training step'); acc_ax.set_ylabel('Accuracy')        
        
        
        if baseline_metrics is not None:
            # optionally show the training curves of a previous metric object as baseline
            # (by extracting the line data from the plots saved to that object)
            baseline_train_x = baseline_metrics.plot_axes[0].lines[0].get_xdata()
            baseline_train_loss = baseline_metrics.plot_axes[0].lines[0].get_ydata()
            baseline_val_x = baseline_metrics.plot_axes[0].lines[1].get_xdata()
            baseline_val_loss = baseline_metrics.plot_axes[0].lines[1].get_ydata()
            baseline_train_acc = baseline_metrics.plot_axes[1].lines[0].get_ydata()
            baseline_val_acc = baseline_metrics.plot_axes[1].lines[1].get_ydata()

            loss_ax.plot(baseline_train_x, baseline_train_loss, c='lightgray', linestyle=':', label=None, zorder=-1)
            loss_ax.plot(baseline_val_x, baseline_val_loss, c='lightgray', linestyle='-', label=None, zorder=-1)
            acc_ax.plot(baseline_train_x, baseline_train_acc, c='lightgray', linestyle=':', label=None, zorder=-1)
            acc_ax.plot(baseline_val_x, baseline_val_acc, c='lightgray', linestyle='-', label=None, zorder=-1)
            baseline_end_step = max(baseline_train_x)
        else:
            baseline_end_step = 0

        # format accuracy y-axis as percentage:
        acc_ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1.0))
        acc_ax.yaxis.tick_right()
        acc_ax.yaxis.set_label_position('right')

        # and place x-axis minor tick marks at epoch boundaries:
        loss_ax.set_xticks(self.epoch_steps, minor=True)
        acc_ax.set_xticks(self.epoch_steps, minor=True)
    
        # draw line/s for comparison with baseline if given:
        if baseline_accs is not None:
            for b_name, b_acc in baseline_accs.items():
                acc_ax.axhline(b_acc, c=[0.8]*3, linestyle=':')
                # add text label as well:
                acc_ax.text(0, b_acc+0.003, b_name, c=[0.6]*3, size=8)

        # and to demarcate self-training iterations:
        if len(self.iter_steps) > 0:
            for iter_step in self.iter_steps:
                loss_ax.axvline(iter_step, c=[0.8]*3, linestyle='--')
                acc_ax.axvline(iter_step, c=[0.8]*3, linestyle='--')
                

        # add epoch tick markers and legend:
        end_step = max(max_step, self.step, baseline_end_step)
        
        num_epochs = len(self.epoch_steps)
        if num_epochs > 1:
            num_expected_epochs = end_step // self.epoch_steps[1]
            tick_spacing = max(num_expected_epochs // 10, 1)
        else:
            tick_spacing = 1
            num_expected_epochs = 1
        # loss_ax.set_xticks(self.epoch_steps[::tick_spacing], range(0, num_epochs, tick_spacing))
        # loss_ax.tick_params(axis='both', which='minor', labelsize=6)
        loss_ax.legend(); loss_ax.set_xlabel('Training Step'); loss_ax.set_ylabel('Loss')
        
        # set x lims based on max number of steps
        loss_ax.set_xlim([0-(end_step*0.05), end_step*1.05])
        acc_ax.set_xlim([0-(end_step*0.05), end_step*1.05])
        # acc_ax.set_xticks(self.epoch_steps[::tick_spacing], range(0, num_epochs, tick_spacing))
        # acc_ax.tick_params(axis='both', which='minor', labelsize=6)
        
        acc_ax.legend(); acc_ax.set_xlabel('Training Step'); acc_ax.set_ylabel('Accuracy (%)')       
        # and y lims based on the expectation that loss goes down and acc goes up
        loss_ax.set_ylim([0, None])
        acc_ax.set_ylim([None, 1])

        
        self.fig.suptitle(title)
        plt.tight_layout()

        if live:
            # redraw using ipython display
            display(self.fig)
            clear_output(wait=True)
        else:
            # just show plot normally
            plt.show()


### function to verify the contents and labels of a dataset object:
def inspect_dataset(data, layout=(3,3)):
    """accepts a LabelledDataset or UnlabelledDataset object
    and plots images from the data with optional class annotations"""
    num_examples = np.prod(layout)
    example_idxs = np.random.choice(range(len(data)), num_examples, replace=False)

    images = [data.images[idx] for idx in example_idxs]

    if hasattr(data, 'targets') and data.classes is not None:
        # show labels for classification datasets
        show_labels = True
        labels = [data.targets[idx] for idx in example_idxs]
    else:
        show_labels = False

    a = 0
    fig, axes = plt.subplots(*layout)
    for r, row in enumerate(axes):
        for c, ax in enumerate(row):
            image = images[a]
            # image, label = examples[i]
            ax.imshow(images[a])
            
            if show_labels:
                label = f'{labels[a]}: {data.classes[labels[a]]}'
                ax.set_title(label)
            
            # tidy up axis:
            ax.set_axis_off()
            a += 1

    fig.suptitle(data.name)
    fig.tight_layout()
    
    plt.show()        



### function for inspecting the correctness of the outputs of a model:
def inspect_batch(images: torch.Tensor, # batch of images as torch tensors
        labels: torch.Tensor=None,      # optional vector of ground truth label integers
        predictions: torch.Tensor=None, # optional vector/matrix of model predictions
        # display parameters:
        class_names: list[str]=None, # optional list or dict of class idxs to class name strings
        title: str=None,       # optional title for entire plot    
        # figure display/sizing params:
        center_title=True,
        max_to_show=15,
        num_cols = 5,
        scale=0.6,
        ):
    """accepts a batch of images as a torch tensor or list of tensors,
    and plots them in a grid for manual inspection.
    optionally, you can supply ground truth labels
    and/or model predictions, to display those as well."""

    max_to_show = min([max_to_show, len(images)]) # cap at number of images
    
    num_rows = int(np.ceil(max_to_show / num_cols))

    # add extra figure height if needed for captions:
    extra_height = (((labels is not None)*0.4 + (predictions is not None))*0.4)

    fig_width = 2 * scale * num_cols
    fig_height = (2+extra_height) * scale * num_rows + ((title is not None) * 0.4)
    
    fig, axes = plt.subplots(num_rows, num_cols, squeeze=False, figsize=(fig_width, fig_height))
    all_axes = []
    for ax_row in axes:
        all_axes.extend(ax_row)

    # translate labels and predictions to class names if given:
    if class_names is not None:
        if labels is not None:
            labels = [f'{class_names[int(l)]}' for l in labels]
        if predictions is not None:
            if len(predictions.shape) == 2:
                # probability distribution or onehot vector, so argmax it:
                pred_idxs = predictions.argmax(dim=1)
                pred_classes = [f'{class_names[int(i)]}' for i in pred_idxs]
                confs = F.softmax(predictions, dim=1).max(dim=1)[0]
            pred_strs = [f'{pred_classes[p]} ({confs[p]:.0%})' for p in range(len(predictions))]
    
    for b, ax in enumerate(all_axes):
        if b < max_to_show:
            # rearrange to H*W*C:
            img_p = images[b].permute([1,2,0]) 
            # un-normalise:
            img = (img_p - img_p.min()) / (img_p.max() - img_p.min())
            # to numpy:
            img = img.cpu().detach().numpy()
            
            ax.imshow(img)
            ax.axes.get_xaxis().set_ticks([])
            ax.axes.get_yaxis().set_ticks([])

            if labels is not None:                    
                ax.set_title(f'{labels[b]}', fontsize=10*scale**0.5)
            if predictions is not None:
                ax.set_title(f'{pred_strs[b]}', fontsize=10*scale**0.5)
            if labels is not None and predictions is not None:
                if labels[b] == pred_classes[b]:
                    ### matching prediction, mark as correct:
                    mark, color = '✔', 'green'
                else:
                    mark, color = '✘', 'red'
                
                ax.set_title(f'label:{labels[b]}    \n{pred_strs[b]} {mark}', color=color, fontsize=8*scale**0.5)
        else:
            ax.axis('off')
    if title is not None:
        if center_title:
            x, align = 0.5, 'center'
        else:
            x, align = 0, 'left'
        fig.suptitle(title, fontsize=14*scale**0.5, x=x, horizontalalignment=align)
    fig.tight_layout()
    plt.show()



### and a helper function for simple class prediction accuracy:
def top1_acc(pred, y):
    """calculates accuracy over a batch as a float
    given predicted logits 'pred' and integer targets 'y'"""
    return (pred.argmax(axis=1) == y).float().mean().item()  

## Data

Since for this exercise, we'll need to do some engineering around Datasets, Dataloaders and batch collate functions, in either case we'll use a custom dataset class as a wrapper for whatever we choose. Below I define two lightweight Dataset subclasses - one for unlabelled, and one for labelled data. You can modify them later if you need to:

In [ ]:
class UnlabelledDataset(Dataset):
    """Simple dataset subclasses that stores only images, no labels"""
    def __init__(self, 
                 images: list, # of PIL.Image or np.ndarray
                 name: str, # name of dataset
                ): 
        self.images = images
        self.name = name
    
    def __getitem__(self, i: int):
        img = self.images[i]
        # if not already a PIL image instance, try to make it one:
        if not isinstance(img, Image):
            img = PIL.Image.fromarray(img)
        return img

    def __len__(self):
        return len(self.images)

    def inspect(self):
        return inspect_dataset(self)        

class LabelledDataset(Dataset):
    """Simple dataset subclass that stores image/label pairs.
    labels can be anything (integer labels, regression targets, matrices, etc.)"""
    def __init__(self, 
                 images:list, # of PIL.Image or np.ndarray
                 targets:list, # of anything
                 name: str, # name of dataset
                 classes:list[str]=None, # optional: for classification datasets, a list or dict of class names
                ):
        assert len(images) == len(targets), "Must have the same number of images as targets"
        self.images = images
        self.targets = targets

        self.name = name
        self.classes = classes
        self.num_classes = len(classes) if classes is not None  else None
    
    def __getitem__(self, i: int):
        img = self.images[i]
        # if not already a PIL image instance, try to make it one:
        if not isinstance(img, Image):
            img = PIL.Image.fromarray(img)
        target = self.targets[i]
        return img, target

    def __len__(self):
        return len(self.images)

    def inspect(self):
        return inspect_dataset(self)        

    def count_labels(self):
        """if this is a classification dataset,
        print out the counts of each individual class"""
        assert self.classes is not None
        label_counts = np.unique(self.targets, return_counts=True)[1]
        print(f'\n{self.name}: {len(self):,} samples')

        if isinstance(self.classes, dict):
            idxs, names = self.classes.keys(), self.classes.values()
        elif isinstance(self.classes, list):
            idxs, names = range(len(self.classes)), self.classes
        for idx, name in zip(idxs, names):
            num_samples = label_counts[idx]
            print(f'  {idx}: {name:<13} ({num_samples:,} samples)')

    def train_test_split(self, 
                         test_frac: float, # between 0-1
                         seed=0): # for reproducibility
        """Splits this dataset into two train/test subsets"""
        np.random.seed(seed)
        num_test_samples = int(len(self) * test_frac)
        test_idxs = np.random.choice(len(self), size=num_test_samples, replace=False)
        train_idxs = [i for i in range(len(self)) if i not in test_idxs]

        train_split = LabelledDataset([self.images[i] for i in train_idxs],
                                      [self.targets[i] for i in train_idxs],
                                      name = f'TRAIN_{self.name}',
                                      classes = self.classes)
        test_split = LabelledDataset([self.images[i] for i in test_idxs],
                                      [self.targets[i] for i in test_idxs],
                                      name = f'VAL_{self.name}',
                                      classes = self.classes)
        return train_split, test_split

### Dataset Selection

We define our unlabelled and labelled datasets below.

I give you a choice of two options:
1. The usual CIFAR-100-subset from previous exercise as target (labelled) data, and CIFAR-10 with its labels destroyed as the source data.
2. The STL-10 dataset, which inherently contains a labelled and unlabelled portion, with similar distributions but distinct classes.

STL-10 is very similar to CIFAR-10:
- it has 10 classes, and they are very similar to the CIFAR ones: airplane, bird, car, cat, deer, dog, horse, monkey, ship, truck.
- Unlike CIFAR-10, it's at a higher resolution, (96x96 instead of 32x32) so it will take more time to train (unless you resize it)
In addition, it includes a large amount (100,000) of unlabelled images:
- At the same resolution, and from a similar distribution, but not the same classes.
- That is, the unlabelled images include houses, lizards, shopping carts, and so on.
- This is a realistic semi-supervised setting - we have some labels, but we can't confirm that every example in our unlabelled set actually maps to one of the labels in our labelled set.

Both will work similarly for this task.  it's harder on the GPU and will take more time to train (unless you resize it).

I think the **STL-10 data** is **more fun** to work with for this task, but it's also a larger download (about 10x heavier) and will train much more slowly, so if you want **to save time**, use **CIFAR**. Instead, you may find it interesting to develop your task on CIFAR as a prototype, and then do a longer self-supervised training run on STL in your own time, but it's totally up to you. 

In [ ]:
#           choose:
##### 'cifar' or 'stl10'  #####
base_data = 'cifar'
###############################

data_root = 'data/' # or wherever you want to put the data files

img_size = 96 if base_data=='stl10' else 32

In [ ]:
if base_data == 'cifar':
    cifar_superclasses = {
       "aquatic mammals": ["beaver", "dolphin", "otter", "seal", "whale"],
                  "fish": ["aquarium_fish", "flatfish", "ray", "shark", "trout"],
               "flowers": ["orchid", "poppy", "rose", "sunflower", "tulip"],
       "food containers": ["bottle", "bowl", "can", "cup", "plate"],
         "fruit and veg": ["apple", "mushroom", "orange", "pear", "sweet_pepper"],
           "electricals": ["clock", "keyboard", "lamp", "telephone", "television"],
             "furniture": ["bed", "chair", "couch", "table", "wardrobe"],
               "insects": ["bee", "beetle", "butterfly", "caterpillar", "cockroach"],
      "large carnivores": ["bear", "leopard", "lion", "tiger", "wolf"],
       "large buildings": ["bridge", "castle", "house", "road", "skyscraper"],
        "natural scenes": ["cloud", "forest", "mountain", "plain", "sea"],
      "large herbivores": ["camel", "cattle", "chimpanzee", "elephant", "kangaroo"],
        "medium mammals": ["fox", "porcupine", "possum", "raccoon", "skunk"],
         "invertebrates": ["crab", "lobster", "snail", "spider", "worm"],
                "people": ["baby", "boy", "girl", "man", "woman"],
              "reptiles": ["crocodile", "dinosaur", "lizard", "snake", "turtle"],
         "small mammals": ["hamster", "mouse", "rabbit", "shrew", "squirrel"],
                 "trees": ["maple_tree", "oak_tree", "palm_tree", "pine_tree", "willow_tree"],
              "vehicles": ["bicycle", "bus", "motorcycle", "pickup_truck", "train"],
        "weird vehicles": ["lawn_mower", "rocket", "streetcar", "tank", "tractor"]
    }
    
    ##### choose your favourite: #####
    TARGET_CATEGORY = 
    ##################################


    def download_from_mirror(gdown_uri, dataset_name):
        tar_path = os.path.join(data_root, dataset_name + '.tar.gz')
        if not os.path.exists(data_root):
            os.makedirs(data_root)
        if not os.path.exists(tar_path):
            # fetch dataset .tar.gz:
            gdown.download(id=gdown_uri, output=tar_path, quiet=False)
            print(f'Data tarball downloaded successfully: {tar_path}')
        else:
            print(f'Detected existing tarball: {tar_path}')
        # unpack raw data from archive:
        data_dir = os.path.join(data_root, dataset_name)
        if not os.path.exists(data_dir):
            datasets.utils.extract_archive(tar_path, data_root)
            print(f'{dataset_name} extracted successfully.')
    

    
    ### download CIFAR-100 data:
    cifar100_uri = '1MF3kdv_CYZnM-89B5AsB_KiAbObAamIG'
    download_from_mirror(cifar100_uri, 'cifar-100-python')    
    
    cifar100 = datasets.CIFAR100(root=data_root, download=False)
    # then we create the subset:
    superclass_names = cifar_superclasses[TARGET_CATEGORY]
    superclass_idxs = set([cifar100.class_to_idx[n] for n in superclass_names])
    subset_idxs = [i for i,idx in enumerate(cifar100.targets) if idx in superclass_idxs]

    subset_imgs = cifar100.data[subset_idxs]
    subset_old_labels = [cifar100.targets[i] for i in subset_idxs]

    # remap from original cifar100 idxs (0-99) to new clean (0-4)
    class_remap = {cifar100.class_to_idx[name]: i for i,name in enumerate(superclass_names)}
    # old_targets = [label for img,label in target_data]
    subset_labels = [class_remap[t] for t in subset_old_labels]
    
    target_data = LabelledDataset(subset_imgs, 
                                  subset_labels, 
                                  name = f'Target Data: CIFAR-{TARGET_CATEGORY.title()}', 
                                  classes = superclass_names)
    del cifar100
    target_data.count_labels()
    target_data.inspect()
    

    
    ### download CIFAR-10 data:
    cifar10_uri = '1Mo5oi1MD6XIHFekqGPDSGl66MiPbhl99'
    download_from_mirror(cifar10_uri, 'cifar-10-python')
    
    cifarx = datasets.CIFAR10(root=data_root)
    # and destroy all labels:
    cifarx.targets = [-1]*len(cifarx.data)
    # keep the images only:
    source_imgs = cifarx.data
    source_data = UnlabelledDataset(source_imgs,
                                   name = 'Source Data: CIFAR-X')
    del cifarx
    print(f'{source_data.name}: {len(source_data)} samples')
    source_data.inspect()

else:
    assert base_data == 'stl10', "base_data must be one of: ('cifar', 'stl10')"

In [ ]:
### download the stl-10 data if required: 
if base_data == 'stl10':

    if not os.path.exists(data_root):
        os.makedirs(data_root)
    
    # check for archive file if you downloaded it separately
    if 'stl10_binary.tar.gz' in os.listdir(data_root) and 'stl10_binary' not in os.listdir(data_root):
        print('Extracting STL-10 binary from archive: stl10_binary.tar.gz')
        datasets.utils.extract_archive(os.path.join(data_root, 'stl10_binary.tar.gz'), data_root)

    # optionally: resize images to a smaller dimension for easier training
    RESIZE_STL = True
    if RESIZE_STL:
        resize_dims = (64, 64)
        resize_transform = transforms.Resize(resize_dims)
    else:
        resize_transform = None
    
    # download the data, split by labelled and unlabelled fraction
    stl10_lab = datasets.STL10(root=data_root, split='train', transform=resize_transform, download=True)
    target_imgs = stl10_lab.data.transpose([0,2,3,1]) # rearrange CHW to HWC format
    target_labels = stl10_lab.labels
    target_data = LabelledDataset(target_imgs, target_labels,
                                  name = 'Target Data: STL-10-Labelled',
                                  classes = stl10_lab.classes)
    del stl10_lab

    target_data.count_labels()
    target_data.inspect()

    stl10_unlab = datasets.STL10(root=data_root, split='unlabeled', transform=resize_transform, download=False)
    source_data = UnlabelledDataset(stl10_unlab.data.transpose([0,2,3,1]),
                                   name = 'Source Data: STL-10-Unlabelled')

    del stl10_unlab
    print(f'{source_data.name}: {len(source_data):,} samples')
    source_data.inspect()


In [ ]:
### create a train/validation split for the target data:

val_frac = 0.1
target_data_train, target_data_val = target_data.train_test_split(val_frac, seed=0)
for split, split_name in zip([target_data_train, target_data_val],
                       ['train', 'val']):
    print(f'{target_data.name} | {split_name:>5} split: {len(split):,} examples')

In [ ]:
### set up preprocessing and data loaders for baseline training:

batch_size = 128 if base_data == 'cifar' else 32  # STL-10 is bigger, so we compensate with smaller batches
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


normalise = transforms.Normalize(mean=0.5, std=0.2,)

preprocess = transforms.Compose([
        transforms.ToTensor(),
        normalise,
    ])


# if needed, a torch transform to apply some strong augmentations to an image:
random_augment = transforms.Compose([
            transforms.ToTensor(), #
            transforms.RandomResizedCrop(size=img_size, scale=(0.5, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([
                transforms.ColorJitter(0.5, 0.5, 0.5, 0.2)
            ], p=0.5),
            transforms.RandomErasing(p=0.5, scale=(0.01, 0.25))
        ])

preprocess_aug = transforms.Compose([
        random_augment,
        normalise,
    ])


def collate_labelled(batch, aug=False):
    images = [b[0] for b in batch]
    labels = [b[1] for b in batch]

    transform = preprocess if not aug   else preprocess_aug
    
    image_tensor = torch.stack([transform(img) for img in images])
    label_tensor = torch.tensor(labels, dtype=torch.long)
    
    return image_tensor.to(device), label_tensor.to(device)

# collate function for supervised training with data augmentation
collate_labelled_aug = lambda x: collate_labelled(x, aug=True)
    

def collate_unlabelled(batch, aug=False):
    images = [b[0] if isinstance(b, tuple) else b  for b in batch]

    transform = preprocess if not aug   else preprocess_aug
    
    image_tensor = torch.stack([transform(img) for img in images])
    
    return image_tensor.to(device)

collate_unlabelled_aug = lambda x: collate_unlabelled(x, aug=True)
    

# train/val loaders for supervised training on target set:
target_train_loader = DataLoader(target_data_train, 
                                 batch_size=batch_size,
                                 collate_fn=collate_labelled,
                                 shuffle = True)

# with data augmentation:
target_train_loader_aug = DataLoader(target_data_train, 
                                 batch_size=batch_size,
                                 collate_fn=collate_labelled_aug,
                                 shuffle = True)

target_val_loader =   DataLoader(target_data_val, 
                                 batch_size=batch_size,
                                 collate_fn=collate_labelled,
                                 shuffle=False)

# unsupervised data loader on source set if you need it:
source_unsup_loader = DataLoader(source_data,
                                batch_size=batch_size,
                                collate_fn=collate_unlabelled,
                                shuffle=True)

# pull an example batch:

aug_batch_x, aug_batch_y = next(iter(target_train_loader_aug))
inspect_batch(aug_batch_x, 
              aug_batch_y, 
              class_names=target_data.classes,
             title=f'Example batch ({target_data_train.name}), augmented')

val_batch_x, val_batch_y = next(iter(target_val_loader))
inspect_batch(val_batch_x, 
              val_batch_y, 
              class_names=target_data.classes,
             title=f'Example batch ({target_data_val.name})')


u_batch_x = next(iter(source_unsup_loader))
inspect_batch(u_batch_x, 
             title=f'Example batch ({source_data.name})')

## Models

Here we use a similar class to the BasicCNN classifier we were training before, but we abstract out the Backbone as its own class (which outputs 'features' to be reused later by a ClassificationHead class)

In [ ]:

class ConvBackbone(nn.Module):
    def __init__(self, 
                 num_features: int, # how many output features in the final layer
                 input_channels=3, # dimensionality of input (i.e. 3 for RGB images)
                 dropout=0.3,      # dropout probability after MLP layers
                 device=device,    # 'cuda' or 'cpu'
                ):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2) # after first two conv layers

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        
        self.fc1 = nn.Linear(256, 256)
        self.fc2 = nn.Linear(256, 128)

        self.dropout = nn.Dropout(dropout)
        
        self.features = nn.Linear(128, num_features)

        self.to(device)
    
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.relu(self.conv3(x))
        x = torch.relu(self.conv4(x))
        
        x = self.avg_pool(x)
        x = torch.flatten(x, 1)

        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        
        x = torch.relu(self.features(x))

        return x


backbone_output_size = 64

# if backbone_type == 'resnet':
def initialise_backbone():
    backbone = ConvBackbone(num_features=backbone_output_size)
    return backbone.to(device)

baseline_backbone = initialise_backbone()

print(summary(baseline_backbone, 
        input_size=val_batch_x.shape,
        col_names=("input_size", "output_size", "num_params"),
       ))


This time, for the model class, we'll use a slightly different abstraction. Since we will be adapting the model for different tasks, we will call the base model the 'backbone', and keep it as a modular unit we can combine with a task-specific classification head.

Below, I create a little Module subclass for this classification head, which contains a linear projection layer followed by the final layer for classification. After that, I show an example of how to attach it to our backbone to create the baseline model.

In [ ]:
### define the classification head that we attach to the encoder backbone:
class ClassifierHead(nn.Module):
    def __init__(self, input_size: int, 
                 projection_size: int, 
                 num_classes: int,):
        super().__init__()

        self.projection = nn.Linear(input_size, projection_size)
        self.classifier = nn.Linear(projection_size, num_classes)

        self.to(device)

    def forward(self, x):
        # assume x is already activated feature logits,
        x = self.projection(x)
        x = self.classifier(F.relu(x))

        return x

In [ ]:
### create the BASELINE MODEL as a concatenation of the backbone and the task head:

num_target_classes = target_data.num_classes

target_task_head = ClassifierHead(input_size = backbone_output_size,
                                  projection_size = 32, # some reasonable number between backbone_output_size and num_target_classes
                                  num_classes = num_target_classes)

# chain the two modules together:
baseline_model = nn.Sequential(baseline_backbone, target_task_head)

from torchinfo import summary
print(summary(baseline_model,
        input_size=val_batch_x.shape, # batch, channels, height, width
        col_names=("output_size", "num_params"),
       ))


## Baseline Training

Below is the usual supervised training function.

In [ ]:
### simple model training loop:

def train_supervised(model: nn.Module, 
                train_loader: torch.utils.data.DataLoader,
                val_loader: torch.utils.data.DataLoader,
                num_epochs: int, 
                lr:float,  # initial learning rate
                l2_reg:float = 0.0, # L2 weight decay term (0 means no regularisation)

                baseline_accs: dict[str:float]=None, # dict of name:baseline accuracy benchmarks to compare against
                baseline_metrics: TrainingMetrics=None, # optional metrics object to plot against
                dynamic_plot:bool = True, # whether to show live updating training plot
                title: str = '', # optional title for plot/metrics
                seed: int = None, # fixed seed for repeatable results
                early_stopping: bool = False, # if True, interrupt training when model converges
               ):
    """trains a model object in-place on a provided train_loader object,
    validating on a provided val_loader object,
    and compares with provided TrainingMetrics object/s as baseline if desired.
    returns TrainingMetrics object for plotting"""
    
    # define loss and optimiser:
    loss_func = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=l2_reg)

    metrics = TrainingMetrics()
    
    if seed is not None:
        torch.manual_seed(seed)
        
    for e in range(num_epochs):
        model.train()
        for batch in train_loader:
            x, y = batch

            opt.zero_grad()      
            # forward pass:
            pred = model(x)
            batch_loss = loss_func(pred, y)

            # backprop:
            batch_loss.backward()
            opt.step()
    
            # track training loss and accuracy at every batch:
            batch_acc = top1_acc(pred, y)
            metrics.log_train(loss=batch_loss.item(), acc=batch_acc)

        
        # evaluate after each epoch::
        val_loss, val_acc = evaluate_model(model, val_loader, loss_func)
        metrics.log_val(loss=val_loss, acc=val_acc)
        
        if dynamic_plot:
            metrics.plot(title=title, live=True,
                         max_step = len(train_loader)*num_epochs,
                         baseline_accs=baseline_accs,
                         baseline_metrics=baseline_metrics)
        else:
            epoch_train_loss, epoch_val_loss = metrics.epoch_loss
            epoch_train_acc, epoch_val_acc = metrics.epoch_acc
            print(f'epoch:{e:>3} | loss: {epoch_train_loss:.2f} | {epoch_val_loss:.2f}   ' +
              f'acc: {epoch_train_acc:.1%} | {epoch_val_acc:.1%}')

        if early_stopping and (e>4):
            # interrupt training if val accuracy has not decreased in 3 epochs:
            recent_val_loss = np.mean(metrics.val_loss[-4:-1])
            if metrics.val_loss[-1] > recent_val_loss:
                print(f'Validation loss converged after {e+1} epochs, training stopped')
                break

            
    # plot training curve at end if we don't already have it:
    if not dynamic_plot:
        metrics.plot(title=title, live=False,
                     baseline_accs=baseline_accs,
                     baseline_metrics=baseline_metrics)
    
    return metrics


def evaluate_model(model: nn.Module, 
                   val_loader: torch.utils.data.DataLoader,
                   loss_func: nn.Module = nn.CrossEntropyLoss()):
    """Runs model in eval mode over a DataLoader of validation data
    and returns aggregated validation loss and accuracy"""
    model.eval()
    with torch.no_grad():
        # make lists to aggregate batch metrics across validation set:
        batch_val_losses, batch_val_accs = [], []

        for batch in val_loader:
            vx, vy = batch

            # run val batch on model:
            vpred = model(vx)
            # calculate batch-wise metrics:
            val_loss = loss_func(vpred, vy)
            val_acc = top1_acc(vpred, vy)
            
            batch_val_losses.append(val_loss.item())
            batch_val_accs.append(val_acc)
    # aggregate and return dataset-level metrics:
    val_loss = np.mean(batch_val_losses)
    val_acc = np.mean(batch_val_accs)
    return val_loss, val_acc

### and a helper function for simple class prediction accuracy:
def get_batch_acc(pred, y):
    """calculates accuracy over a batch as a float
    given predicted logits 'pred' and integer targets 'y'"""
    return (pred.argmax(axis=1) == y).float().mean().item()

Now we train the baseline model. Since the training set is so small, it's very fast and overfits very quickly, so we'll do only a few epochs at a very small learning rate:

In [ ]:
### training hyperparameters

ft_num_epochs = 30
ft_lr = 1e-3 # initial learning rate
ft_l2_reg = 1e-1 # L2 loss term


# re-initialise backbone and task head:
baseline_backbone = initialise_backbone()
baseline_target_task_head = ClassifierHead(input_size = backbone_output_size,
                                  projection_size = 32, # some reasonable number between 1000 and num_target_classes
                                  num_classes = num_target_classes)
baseline_model = nn.Sequential(baseline_backbone, baseline_target_task_head)


print(f'\nTraining baseline supervised model on {target_data.name}:')

print(summary(baseline_model,
        input_size=val_batch_x.shape, # batch, channels, height, width
        col_names=("output_size", "num_params"),
       ))


In [ ]:
baseline_metrics= train_supervised(baseline_model, 
                                target_train_loader, target_val_loader, 
                                ft_num_epochs, ft_lr, ft_l2_reg,
                                title=f'Baseline training ({target_data.name})',
                                seed=0)

# baseline_metrics =    
baseline_acc = baseline_metrics.best_val_acc
print(f'Baseline (supervised only) val acc: {baseline_acc:.1%}')

Probably you will see that the model overfits very quickly. This is our baseline for the performance achievable by training on the supervised labels only.

You can play around with the hyperparameters to try and get stable performance if you want; just make sure to use the same hyperparameters for the fine-tuning later.

Your aim is to improve this performance using self-supervised learning. 

## Task 1: Pretext Training

**Your task** is to improve the performance of the supervised model, by creating a pretext task and pre-training the backbone on that, then fine-tuning on the supervised labels again.

The key component of this is defining the pretext task over the unsupervised data, in a way that will encourage the backbone to learn useful features that will help in the downstream supervised task.

The choice of pretext task is up to you. You must implement it!

#### Pretext Task Ideas
You can refer back to the lecture for a discussion of good pretext tasks, but here are some suggestions:
- Apply a four-way rotation and predict the angle as a classification
- Apply a colour jitter and predict the intensity of the transformation as a regression
- Apply a crop and predict the parameters of the crop
- Provide two cropped image patches and predict their relative position
- Split an image into multiple patches and predict their arrangement (the jigsaw task)
- Provide two cropped/augmented images, predict whether they are crops of the same image

The important thing is to make a pretext task that is **not too easy**. For example, a de-noising or de-blurring task sounds like an OK idea, but a model can learn to perform de-noising using only low-level features - so it would probably not learn the high-level semantic features we want, and therefore won't transfer well to supervised fine-tuning on the target data.

You could attempt a task like in-painting masked images, or the full [SimCLR](https://theaisummer.com/simclr/) similarity matrix, but these are more complex to implement and I wouldn't recommend it if you want to get things done within a couple hours!

An alternative to the SimCLR loss if you want to do contrastive learning, is the simpler **pairwise contrastive loss**, or the related **triplet loss** (see [here](https://lilianweng.github.io/posts/2021-05-31-contrastive/))

Full pixel generative tasks in particular are tricky to optimise; but you could consider training on **image residuals** instead 
- e.g. for image recolourising or undistorting - instead of outputting the coloured image directly, you can train to output the **pixel-wise difference** between the augmented and the original image, which is conceptually an equivalent task but trains much more smoothly.

Whichever pretext task you choose, **applying random augmentations** helps a lot in training generalisable features that will help your downstream performance. It's almost always necessary. So, if you're doing image rotation, you could rotate the images (as the pretext signal) and then apply a random crop/blur/colour distortion (for stability).
- But be careful - choose augmentations that *don't conflict with the pretext task* you've set! For example, if you are predicting four-way rotation, your augmentations should not include a random rotation on top.


### Step-by-step
You'll need to:
1. Choose a pretext task that you can implement
2. Implement the pretext task by writing whatever transform functions you need to produce the correct images and labels (see [torchvision.transforms](https://docs.pytorch.org/vision/main/transforms.html))
3. Format the data for the pretext task...
  - Either by creating a new dataset with specific images and labels, and wrapping those in a `LabelledDataset`
  - or by using your transformations at batch-time, pulling unsupervised samples from `source_data` and computing the correct images/labels at each pass
  - You'll probably need to create a new `collate_fn` for a new data loader that suits your task. You can base this on the example used earlier.
  - And consider whether you need to create a training/validation split for this pretext task as well
4. Initialise a model for pre-training on your pretext task, and train it.
  - You'll create a backbone (see `initialise_backbone()` above) and a specific `ClassifierHead` that fits your task.
  - If you need to create a new training loop, you can adapt `train_supervised` for your needs.
5. Create a model with for training on the original target task (as we did for `baseline_model`)
  - and transfer the trained backbone to the new model (same as in the Transfer Learning practical - I recommend saving the pre-trained backbone's state dict and loading it into the new model)
6. Fine-tune your model on the target task and compare to baseline performance.

You already know how to do most of these from previous practicals - feel free to copy your code from before. The new part is the engineering around designing the pretext task itself.


### Tips

- Your pre-training model should converge stably on the pretext task, but we *don't* necessarily want to see very high accuracy - this might indicate that the task is too easy, and the model isn't learning good representations. If you see this, consider making the task harder or applying stronger augmentations.

- Note that many of these pretext tasks are easily optimised as a normal supervised training loop, once you have a way of producing the input-label pairs. You can even re-use the `train_supervised` loop above without modification.
  - For regression or image-to-image tasks, you'll probably need to change the loss function. `nn.CrossEntropyLoss` expects integer labels, but you can use other loss functions for pixel-wise classification or whatever.
- But make sure you're setting up your data loader and respective collate functions correctly if you do this. We already have `source_unsup_loader` which pulls unlabelled batches from `source_data`, but you probably want to make your own loader that pulls images, applies transformations on them, and returns the correct labels for self-supervised training.
- Other tasks might take two images as input, which would involve some more engineering of the training loop. For example, you might have to feed two image patches through the backbone, compute something based on the combination or difference of the features, and send that to your classifier.


I've provided a useful starting point below: some torchvision transforms for random augmentation, and a function to display torch batches as images so you can visually verify your transformations are working as expected.

In [ ]:
# example random augmentation function you can run on image batches:

from torchvision.transforms import v2

# a torch transform to apply some crazy augmentations to an image:
random_augment = v2.Compose([
            v2.RandomResizedCrop(size=img_dim, scale=(0.8, 1.0)),
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomApply([
                v2.ColorJitter(0.8, 0.8, 0.8, 0.2)
            ], p=0.5),
            v2.RandomGrayscale(p=0.2),
            normalise,
        ])
# this can replace pre_transform in your collate_fn, or you can use parts of it for your
# pretext task transformations, or whatver non-pretext augmentations you need




In [ ]:
### implement your pretext task here!



...





If your pretext task works well, you should see your model gradually converging to a stable accuracy. The accuracy might not be very high for your chosen task, but what's important is that the model is learning something, and hopefully the features that it's forced to learn in order to reduce loss on your task are similar to the features needed for the target task. 

# Task 2: Supervised Fine-tuning

The next step is the same as in the first practical on transfer learning. You'll simply take the backbone of your trained, self-supervised model, and fine-tune it on the downstream target task. Compare the accuracy you get with what was achievable only from the baseline supervised training on the labelled data alone. 

### Tips

- You might find that the self-supervised pretext training doesn't help much until the loss begins to converge. That means you might need to run your pretext task for a very long time, depending on its difficulty.
- Augmentations are very important for the pretext task to ensure sample diversity and input robustness, if your task is too easy consider increasing the strength of the augmentations! Whether to continue augmenting during fine-tuning is up to you, but if you do it during fine-tuning you should do the same during baseline training as well for a fairer comparison.
- You might also find it useful to use different hyperparameters for fine-tuning compared to training from scratch - a lower learning rate, or a higher L2 regularisation term. Consider also layer freezing if you managed to get it working during the transfer learning practical.

In [ ]:
### fine-tune your self-supervised model on the target task here!

...



print(f'\nFine-tuning self-supervised model on {target_data.name}:')
ft_metrics = train_supervised(..., 
                                target_train_loader, target_val_loader, 
                                ft_num_epochs, ft_lr, ft_l2_reg,
                                title=f'Post-SSL fine-tuning ({target_data.name})',
                                 baseline_metrics=baseline_metrics,
                                 baseline_accs = {'supervised baseline': baseline_metrics.best_val_acc},
                                seed=0)
 
ft_acc = ft_metrics.best_val_acc
print(f'Post-SSL finetuning val acc: {ft_acc:.1%}')

if ft_acc > baseline_acc:
    improvement = ft_acc - baseline_acc
    print(f'   BEATS the baseline supervised accuracy of {baseline_acc:.1%}')
    print(f'   (improvement of: {improvement:.1%} points)')


### Assessment

* **Describe your pretext task**, your motivation for it, and how you implemented it.

* Report your results with **a plot or two** comparing the final validation accuracy of the self-supervised, fine-tuned model to the supervised baseline.
    
* Also write **a few sentences** summarising any particular issues you ran into, how you solved them, and any insights you gained about designing your pretext task, the self-supervised transfer learning implementation, or anything else you found interesting.

Good luck! Contact me if you have any problems.